In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = ["Python is great", "FastAPI is fast", "Python FastAPI"]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(texts)

print("Shape:", tfidf_matrix.shape)  # (3, 4) = 3 documents, 4 unique words
print("Type:", type(tfidf_matrix))   # Sparse matrix

# See it as a table
print("\nAs array:")
print(tfidf_matrix.toarray())

# What are the words?
print("\nWords:", vectorizer.get_feature_names_out())


In [ ]:
"""
Simple RAG Demo - Shows difference WITH and WITHOUT RAG using local Llama
Run this to see how RAG improves your LLM's responses
"""

import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from llm import ask_llama



def retrieve_context(question, knowledge_base, top_k=2):
    """Simple RAG retrieval using TF-IDF"""
    if not knowledge_base:
        return []
    
    vectorizer = TfidfVectorizer()
    all_texts = knowledge_base + [question] # knowledge base + one question
    tfidf_matrix = vectorizer.fit_transform(all_texts) #(i,j):TF-IDF score of j-th word in i-th doc
    
    query_vector = tfidf_matrix[-1]# since we put the question at the end
    doc_vectors = tfidf_matrix[:-1]# knowledge base docs
    similarities = cosine_similarity(query_vector, doc_vectors)[0] # in case of multiple questions
    
    top_indices = np.argsort(similarities)[-top_k:][::-1] # get the best 2 indices
    return [knowledge_base[i] for i in top_indices if similarities[i] > 0.1]

# Sample knowledge base (simulate a small document database)
knowledge_base = [
    "Python 3.12 was released in October 2023 with improved error messages.",
    "FastAPI is a modern web framework for building APIs with Python.",
    "Django is a full-stack web framework that includes ORM and admin panel.",
    "RAG stands for Retrieval-Augmented Generation, combining search with LLMs.",
    "Machine learning models require training data and validation sets.",
    "REST APIs use HTTP methods like GET, POST, PUT, and DELETE.",
]

def demo_without_rag(question):
    """Demo WITHOUT RAG - just ask LLM directly"""
    print("\n" + "="*70)
    print("🔴 WITHOUT RAG - Direct LLM Query")
    print("="*70)
    
    prompt = f"Answer this question briefly: {question}"
    
    print(f"Question: {question}")
    print(f"\nPrompt sent to Llama:\n{prompt}")
    print("\n" + "-"*70)
    
    response = ask_llama(prompt)
    print(f"Llama's Response:\n{response}")
    
    return response

def demo_with_rag(question):
    """Demo WITH RAG - retrieve context first, then ask LLM"""
    print("\n" + "="*70)
    print("🟢 WITH RAG - Retrieve Context + LLM Query")
    print("="*70)
    
    # Step 1: Retrieve relevant context
    retrieved_context = retrieve_context(question, knowledge_base) #extract top 2 relevant docs
    
    print(f"Question: {question}")
    print(f"\nRetrieved Context from Knowledge Base:")
    for i, ctx in enumerate(retrieved_context, 1):
        print(f"  {i}. {ctx}")
    
    # Step 2: Build enhanced prompt with context
    context_text = "\n".join(retrieved_context)
    prompt = f"""Use the following context to answer the question:

Context:
{context_text}

Question: {question}

Answer based on the context above:"""
    
    print(f"\nEnhanced Prompt sent to Llama:")
    print(prompt)
    print("\n" + "-"*70)
    
    response = ask_llama(prompt)
    print(f"Llama's Response:\n{response}")
    
    return response

def main():

    print("\nThis demo shows how RAG improves LLM responses by providing context.")
    print("Press Ctrl+C to exit anytime.\n")
    
    # Test questions
    test_questions = [
        "What is FastAPI?",
        "When was Python 3.12 released?",
        "What does RAG stand for?"
    ]
    
    print("\nAvailable knowledge base:")
    for i, doc in enumerate(knowledge_base, 1):
        print(f"  {i}. {doc}")
    
    for question in test_questions:
        input(f"\n\nPress Enter to test question: '{question}'")
        
        # WITHOUT RAG
        response_no_rag = demo_without_rag(question)
        
        input("\nPress Enter to see WITH RAG...")
        
        # WITH RAG
        response_with_rag = demo_with_rag(question)
        
        print("\n" + "="*70)
        print("COMPARISON:")
        print("="*70)
        print(f"Without RAG: {response_no_rag}")
        print(f"With RAG: {response_with_rag}")
        print("Result: More accurate, specific, and grounded responses!")
    
    print("\n\n" + "="*70)
    print("DEMO COMPLETE!")
    print("="*70)
    print("\nKey Takeaway:")
    print("- WITHOUT RAG: Generic answers from training data")
    print("- WITH RAG: Specific answers using your documents/data")
    print("\nUse RAG when you need LLM to access:")
    print("  • Your private documents")
    print("  • Up-to-date information")
    print("  • Domain-specific knowledge")
    print("  • User-specific context")


main()